## Example 1: Registry and Fetch from the Cache

In [ ]:
import hashlib
import os
import pathlib
import tempfile

import pooch

# Example 1: a registry and fetch() from the local cache
print("Example 1: Registry and fetch from the cache")
print(f"pooch {pooch.__version__}")

with tempfile.TemporaryDirectory() as cache:
    data_file = pathlib.Path(cache) / "temperature.csv"
    data_file.write_text("day,temp_c\n1,21.5\n2,22.1\n3,19.8\n")

    # file_hash() uses SHA-256 by default and returns the bare hex digest
    known_hash = pooch.file_hash(str(data_file))
    assert known_hash == hashlib.sha256(data_file.read_bytes()).hexdigest(), known_hash

    pup = pooch.create(
        path=cache,
        base_url="https://example.invalid/data/",
        registry={"temperature.csv": f"sha256:{known_hash}"},
    )
    print(f"  Registry: {pup.registry}")
    assert pup.registry_files == ["temperature.csv"], pup.registry_files
    assert pup.get_url("temperature.csv") == "https://example.invalid/data/temperature.csv"

    # The file is already in the cache and its hash matches, so fetch() returns its path
    # without downloading anything (example.invalid never resolves).
    fetched = pup.fetch("temperature.csv")
    assert os.path.samefile(fetched, data_file), (fetched, data_file)
    print(f"  fetch() returned the cached file {pathlib.Path(fetched).name}")

print("Example 1 completed")

## Example 2: Download and Cache with a Custom Downloader

In [ ]:
import os
import pathlib
import shutil
import tempfile

import pooch

# Example 2: download and cache through a custom downloader
print("Example 2: File retrieval and caching")

demo_files = {
    "data1.csv": "name,age,city\nAlice,25,NYC\nBob,30,LA\n",
    "data2.json": '{"version": 1, "data": [1, 2, 3, 4, 5]}',
    "config.txt": "# Configuration file\ndebug=true\ntimeout=30\n",
}

with tempfile.TemporaryDirectory() as server, tempfile.TemporaryDirectory() as cache:
    for name, content in demo_files.items():
        pathlib.Path(server, name).write_text(content)

    # A downloader is any callable(url, output_file, pooch). This one copies from a local
    # directory, so pooch's download, hash check and caching all run offline.
    downloads = []

    def local_downloader(url, output_file, pup):
        downloads.append(url)
        shutil.copyfile(os.path.join(server, url.rsplit("/", 1)[-1]), output_file)

    pup = pooch.create(
        path=cache,
        base_url="https://example.invalid/v1/",
        registry={name: pooch.file_hash(os.path.join(server, name)) for name in demo_files},
    )

    for name, content in demo_files.items():
        path = pup.fetch(name, downloader=local_downloader)
        assert pathlib.Path(path).read_text() == content, name
        print(f"  {name}: downloaded and verified ({len(content)} bytes)")
    assert downloads == [f"https://example.invalid/v1/{name}" for name in demo_files], downloads

    # Second round: every file is cached with a matching hash, so nothing is downloaded
    for name in demo_files:
        pup.fetch(name, downloader=local_downloader)
    assert len(downloads) == len(demo_files), downloads
    print(f"  Second fetch of {len(demo_files)} files: no new downloads")

print("Example 2 completed")

## Example 3: Hash Verification

In [ ]:
import hashlib
import os
import pathlib
import tempfile

import pooch

# Example 3: hash verification
print("Example 3: Hash verification and file integrity")

sample_bytes = b"This is sample data for hash verification"

with tempfile.TemporaryDirectory() as server, tempfile.TemporaryDirectory() as cache:
    source = pathlib.Path(server, "sample.txt")
    source.write_bytes(sample_bytes)

    # file_hash() takes any hashlib algorithm; registry entries are written "alg:hexdigest"
    for alg in ("md5", "sha1", "sha256"):
        digest = pooch.file_hash(str(source), alg=alg)
        assert digest == hashlib.new(alg, sample_bytes).hexdigest(), alg
        print(f"  {alg}: {digest}")

    def corrupting_downloader(url, output_file, pup):
        pathlib.Path(output_file).write_bytes(sample_bytes + b" (corrupted in transit)")

    pup = pooch.create(
        path=cache,
        base_url="https://example.invalid/",
        registry={"sample.txt": "sha1:" + hashlib.sha1(sample_bytes).hexdigest()},
    )

    # A download whose hash does not match the registry is rejected and never reaches the cache
    try:
        pup.fetch("sample.txt", downloader=corrupting_downloader)
    except ValueError as exc:
        assert "hash" in str(exc).lower(), exc
        print("  Corrupted download rejected: hash mismatch")
    else:
        raise AssertionError("pooch accepted a file whose hash does not match the registry")
    assert not os.path.exists(os.path.join(cache, "sample.txt"))

print("Example 3 completed")

## Example 4: Registry Files and Cache Repair

In [ ]:
import os
import pathlib
import shutil
import tempfile

import pooch

# Example 4: registry files, nested paths and cache repair
print("Example 4: Cache management and utilities")

# os_cache() names the per-user cache directory for a project (it does not create it)
default_cache = pooch.os_cache("pooch-demo")
assert "pooch-demo" in default_cache.parts, default_cache

files = {
    "data/temperature.csv": "day,temp_c\n1,21.5\n",
    "data/pressure.csv": "day,hpa\n1,1013\n",
    "docs/readme.md": "# Demo data\n",
}

with tempfile.TemporaryDirectory() as server, tempfile.TemporaryDirectory() as work:
    for name, content in files.items():
        target = pathlib.Path(server, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content)

    # make_registry() writes one "name hash" line per file under a directory
    registry_file = os.path.join(work, "registry.txt")
    pooch.make_registry(server, registry_file)
    print("  Registry file:")
    for line in pathlib.Path(registry_file).read_text().splitlines():
        print(f"    {line[:60]}...")

    pup = pooch.create(path=os.path.join(work, "cache"), base_url="https://example.invalid/")
    pup.load_registry(registry_file)
    assert sorted(pup.registry) == sorted(files), pup.registry
    for name in files:
        assert pup.registry[name] == pooch.file_hash(os.path.join(server, name)), name

    downloads = []

    def local_downloader(url, output_file, pup):
        downloads.append(url)
        shutil.copyfile(os.path.join(server, url[len(pup.base_url):]), output_file)

    path = pathlib.Path(pup.fetch("data/temperature.csv", downloader=local_downloader))
    assert path.read_text() == files["data/temperature.csv"]
    assert path.relative_to(pup.abspath).as_posix() == "data/temperature.csv"  # nested path kept

    # Edit the cached copy: its hash no longer matches, so the next fetch downloads it again
    path.write_text("edited locally\n")
    path = pathlib.Path(pup.fetch("data/temperature.csv", downloader=local_downloader))
    assert path.read_text() == files["data/temperature.csv"]
    assert len(downloads) == 2, downloads
    print("  An edited cache entry was replaced by a fresh download")

print("Example 4 completed")

## Example 5: retrieve(), Cache Location from the Environment, and Locked Caches

In [ ]:
import os
import pathlib
import tempfile

import pooch

# Example 5: retrieve(), a cache location from the environment, and allow_updates=False
print("Example 5: retrieve, env override and locked caches")

content = b"station,lat,lon\nA,51.5,-0.1\n"

with tempfile.TemporaryDirectory() as work:
    # retrieve() handles a single file without a registry. A cached file with a matching hash
    # is returned as it is, so nothing is downloaded here.
    cached = pathlib.Path(work, "stations.csv")
    cached.write_bytes(content)
    known_hash = pooch.file_hash(str(cached))
    path = pooch.retrieve(
        "https://example.invalid/stations.csv",
        known_hash=known_hash,
        fname="stations.csv",
        path=work,
    )
    assert os.path.samefile(path, cached), path
    print("  retrieve() returned the cached file")

    # env= lets users move the cache without code changes
    override = os.path.join(work, "override")
    os.environ["POOCH_DEMO_DATA_DIR"] = override
    try:
        pup = pooch.create(
            path=pooch.os_cache("pooch-demo"),
            base_url="https://example.invalid/",
            env="POOCH_DEMO_DATA_DIR",
            registry={"stations.csv": known_hash},
        )
    finally:
        del os.environ["POOCH_DEMO_DATA_DIR"]
    assert str(pup.path) == override, pup.path
    print("  Cache path taken from POOCH_DEMO_DATA_DIR")

    # allow_updates=False: a cached file whose hash changed is an error, not a re-download
    os.makedirs(override)
    pathlib.Path(override, "stations.csv").write_bytes(b"stale copy\n")
    locked = pooch.create(
        path=override,
        base_url="https://example.invalid/",
        registry={"stations.csv": known_hash},
        allow_updates=False,
    )
    try:
        locked.fetch("stations.csv")
    except ValueError:
        print("  allow_updates=False refused to replace the stale file")
    else:
        raise AssertionError("fetch() replaced a file although allow_updates=False")

print("Example 5 completed")